In [1]:
# Cell 1: Configure paths (packages pre-installed in pylibs)
import sys

# Packages already installed in pylibs — no pip install needed
sys.path.insert(0, "/local/mnt/workspace/khitesh/MS/pylibs")
print("sys.path configured")

sys.path configured


In [2]:
# Cell 2: Imports
import os, pickle, time

# Redirect HuggingFace cache to workspace (avoids home quota issues)
os.environ["HF_HOME"] = "/local/mnt/workspace/khitesh/MS/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/local/mnt/workspace/khitesh/MS/hf_cache"

import numpy as np
import pandas as pd
import faiss
import fitz  # pymupdf
import anthropic
from sentence_transformers import SentenceTransformer, CrossEncoder

os.makedirs("data", exist_ok=True)
print("All imports OK")

/local/mnt/workspace/khitesh/MS/pylibs/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/local/mnt/workspace/khitesh/MS/pylibs/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


All imports OK


In [3]:
# Cell 3: Download D&D 5e Basic Rules PDF
import urllib.request

PDF_URL = "https://media.wizards.com/2018/dnd/downloads/DnD_BasicRules_2018.pdf"
PDF_PATH = "data/dnd5e_basic_rules.pdf"

if not os.path.exists(PDF_PATH):
    print("Downloading PDF...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print(f"Saved to {PDF_PATH}")
else:
    print(f"PDF already exists at {PDF_PATH}")

PDF already exists at data/dnd5e_basic_rules.pdf


In [4]:
# Cell 4: Extract text from PDF, drop near-empty pages
def extract_pages(pdf_path: str, min_chars: int = 50):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = page.get_text()
        if len(text.strip()) >= min_chars:
            pages.append(text.strip())
    doc.close()
    return pages

pages = extract_pages(PDF_PATH)
print(f"Extracted {len(pages)} non-empty pages")
print("Sample (page 5):\n", pages[4][:300])

Extracted 180 non-empty pages
Sample (page 5):
 5
 Introduction
D&D Basic Rules (Version 1.0). Not for resale. Permission granted to print and photocopy this document for personal use only.
Advantage and Disadvantage
Sometimes an ability check, attack roll, or saving throw 
is modified by special situations called advantage and dis-
advantage. Ad


In [5]:
# Cell 5: Split pages into overlapping chunks
def chunk_pages(pages, chunk_size=400, overlap=50):
    chunks = []
    chunk_id = 0
    for page_num, page_text in enumerate(pages):
        words = page_text.split()
        start = 0
        while start < len(words):
            end = min(start + chunk_size, len(words))
            text = " ".join(words[start:end])
            chunks.append({
                "chunk_id": chunk_id,
                "page": page_num + 1,
                "text": text
            })
            chunk_id += 1
            if end == len(words):
                break
            start = end - overlap  # slide back by overlap
    return chunks

chunks = chunk_pages(pages)
print(f"Total chunks: {len(chunks)}")
print(f"Sample chunk 0: page={chunks[0]['page']}, words={len(chunks[0]['text'].split())}")
print(chunks[0]['text'][:200])

Total chunks: 449
Sample chunk 0: page=1, words=336
1 D&D Basic Rules, Version 1.0, Released November 2018 Credits Lead Designers: Jeremy Crawford, Mike Mearls, Christopher Perkins Design Team: James Wyatt, Rodney Thompson, Robert J. Schwalb, Peter Lee


In [6]:
# Cell 6: Embed all chunks and build FAISS index (cached to disk)
FAISS_PATH = "data/faiss_index.bin"
CHUNKS_PATH = "data/chunks.pkl"

# Load bi-encoder
biencoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(FAISS_PATH) and os.path.exists(CHUNKS_PATH):
    print("Loading cached index and chunks...")
    index = faiss.read_index(FAISS_PATH)
    with open(CHUNKS_PATH, "rb") as f:
        chunks = pickle.load(f)
    print(f"Loaded index with {index.ntotal} vectors, {len(chunks)} chunks")
else:
    print("Embedding chunks (this takes a few minutes on CPU)...")
    texts = [c["text"] for c in chunks]
    embeddings = biencoder.encode(texts, batch_size=64, show_progress_bar=True,
                                  normalize_embeddings=True)
    embeddings = embeddings.astype(np.float32)

    dim = embeddings.shape[1]  # 384
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss.write_index(index, FAISS_PATH)
    with open(CHUNKS_PATH, "wb") as f:
        pickle.dump(chunks, f)
    print(f"Built and saved index: {index.ntotal} vectors, dim={dim}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding chunks (this takes a few minutes on CPU)...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Built and saved index: 449 vectors, dim=384


In [7]:
# Cell 7: Anthropic client (requires ANTHROPIC_API_KEY env var)
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY automatically
MODEL = "claude-haiku-4-5-20251001"

def build_prompt(chunks, query):
    context_parts = [c["text"] for c in chunks]
    context = "\n\n".join(context_parts)
    return (
        "Answer the question using only the following context. "
        "Be precise and cite relevant rules.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )

def call_llm(prompt):
    t0 = time.perf_counter()
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000
    return response.content[0].text, elapsed_ms

In [8]:
# Cell 8: Pipeline 1 — Baseline Retrieval
def run_baseline(query, top_k=10, top_n=3):
    # Retrieval
    t0 = time.perf_counter()
    q_emb = biencoder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, top_k)
    retrieval_ms = (time.perf_counter() - t0) * 1000

    top10 = [
        {**chunks[idx], "bi_rank": rank + 1, "bi_score": float(scores[0][rank])}
        for rank, idx in enumerate(indices[0])
    ]
    top3 = top10[:top_n]

    # Generation
    prompt = build_prompt(top3, query)
    answer, llm_ms = call_llm(prompt)

    return {
        "top10": top10,
        "top3_chunks": top3,
        "answer": answer,
        "retrieval_ms": retrieval_ms,
        "llm_ms": llm_ms,
    }

# Smoke test
result = run_baseline("What is a saving throw?")
print("Baseline answer:\n", result["answer"][:300])
print(f"Retrieval: {result['retrieval_ms']:.1f}ms | LLM: {result['llm_ms']:.1f}ms")

Baseline answer:
 # Saving Throw Definition

According to the context, a saving throw—also called a save—**represents an attempt to resist a spell, a trap, a poison, a disease, or a similar threat.**

## Key Characteristics:

- **When it occurs**: You don't normally decide to make a saving throw; you are forced to ma
Retrieval: 19.8ms | LLM: 4455.3ms


In [9]:
# Cell 9: Pipeline 2 — Cross-Encoder Re-Ranking
crossencoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def run_rerank(query, top_k=10, top_n=3):
    # Step 1: Bi-encoder retrieval (same as baseline)
    t0 = time.perf_counter()
    q_emb = biencoder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, top_k)
    retrieval_ms = (time.perf_counter() - t0) * 1000

    top10 = [
        {**chunks[idx], "bi_rank": rank + 1, "bi_score": float(scores[0][rank])}
        for rank, idx in enumerate(indices[0])
    ]

    # Step 2: Cross-encoder scoring
    t1 = time.perf_counter()
    pairs = [[query, c["text"]] for c in top10]
    ce_scores = crossencoder.predict(pairs)
    rerank_ms = (time.perf_counter() - t1) * 1000

    # Step 3: Re-sort by cross-encoder score descending
    ranked = sorted(
        zip(top10, ce_scores),
        key=lambda x: x[1],
        reverse=True
    )
    top10_ranked = [
        {**chunk, "ce_rank": rank + 1, "ce_score": float(score)}
        for rank, (chunk, score) in enumerate(ranked)
    ]
    top3 = top10_ranked[:top_n]

    # Step 4: Generate answer
    prompt = build_prompt(top3, query)
    answer, llm_ms = call_llm(prompt)

    return {
        "top10_ranked": top10_ranked,
        "top3_chunks": top3,
        "answer": answer,
        "retrieval_ms": retrieval_ms,
        "rerank_ms": rerank_ms,
        "llm_ms": llm_ms,
    }

# Smoke test
result = run_rerank("What is a saving throw?")
print("Re-ranked answer:\n", result["answer"][:300])
print(f"Retrieval: {result['retrieval_ms']:.1f}ms | Rerank: {result['rerank_ms']:.1f}ms | LLM: {result['llm_ms']:.1f}ms")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-ranked answer:
 According to the context provided, a saving throw—also called a save—represents an attempt to resist a spell, a trap, a poison, a disease, or a similar threat. 

To make a saving throw, you roll a d20 and add the appropriate ability modifier (for example, your Dexterity modifier for a Dexterity savi
Retrieval: 32.9ms | Rerank: 397.8ms | LLM: 3255.9ms


In [10]:
# Cell 10: Helper to display rank-shift table and answer comparison
def show_rank_shift(query, br, rr):
    """Display rank-shift table and side-by-side answers for one query."""
    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print(f"{'='*70}")

    # Build rank-shift table: join on chunk_id
    bi_map = {c["chunk_id"]: c for c in br["top10"]}
    ce_map = {c["chunk_id"]: c for c in rr["top10_ranked"]}
    all_ids = list(dict.fromkeys(
        [c["chunk_id"] for c in br["top10"]] +
        [c["chunk_id"] for c in rr["top10_ranked"]]
    ))

    rows = []
    for cid in all_ids:
        bi = bi_map.get(cid, {})
        ce = ce_map.get(cid, {})
        rows.append({
            "chunk_id": cid,
            "page": bi.get("page", ce.get("page")),
            "bi_rank": bi.get("bi_rank", "-"),
            "bi_score": f"{bi.get('bi_score', 0):.4f}",
            "ce_rank": ce.get("ce_rank", "-"),
            "ce_score": f"{ce.get('ce_score', 0):.4f}",
            "text_preview": bi.get("text", ce.get("text", ""))[:80],
        })
    df = pd.DataFrame(rows).sort_values("bi_rank")
    print(df.to_string(index=False))

    print(f"\n--- ANSWER 1 (Baseline) ---\n{br['answer']}")
    print(f"\n--- ANSWER 2 (Re-Ranked) ---\n{rr['answer']}")
    print(f"\nLatency — Baseline: {br['retrieval_ms']:.0f}ms retrieval + {br['llm_ms']:.0f}ms LLM")
    print(f"          Rerank:   {rr['retrieval_ms']:.0f}ms retrieval + {rr['rerank_ms']:.0f}ms rerank + {rr['llm_ms']:.0f}ms LLM")
    return df

print("show_rank_shift helper defined")

show_rank_shift helper defined


In [11]:
# Cell 11.1 — Failure Query 1: Keyword Trap / Distractor
# "Roll a 1" overlaps with "miss"/"fumble" text, but D&D 5e has no critical-fail rule for attacks
QUERY_1 = "What happens when you roll a 1 on an attack roll?"
br1 = run_baseline(QUERY_1)
rr1 = run_rerank(QUERY_1)
df1 = show_rank_shift(QUERY_1, br1, rr1)
latency_rows = []
latency_rows.append({
    "query": "Q1: Roll a 1 on attack",
    "baseline_retrieval_ms": round(br1["retrieval_ms"]),
    "baseline_llm_ms": round(br1["llm_ms"]),
    "baseline_total_ms": round(br1["retrieval_ms"] + br1["llm_ms"]),
    "rerank_retrieval_ms": round(rr1["retrieval_ms"] + rr1["rerank_ms"]),
    "rerank_llm_ms": round(rr1["llm_ms"]),
    "rerank_total_ms": round(rr1["retrieval_ms"] + rr1["rerank_ms"] + rr1["llm_ms"]),
})


QUERY: What happens when you roll a 1 on an attack roll?
 chunk_id  page  bi_rank bi_score  ce_rank ce_score                                                                     text_preview
      206    76        1   0.5392        1   4.1620 76 76 Chapter 9: Combat D&D Basic Rules (Version 1.0). Not for resale. Permissio
      216    79        2   0.3923        4  -8.1294 you instantly. When dam- age reduces you to 0 hit points and there is damage rem
      214    79        3   0.3736        3  -2.5312 79 79 Chapter 9: Combat D&D Basic Rules (Version 1.0). Not for resale. Permissio
      346   130        4   0.3570       10 -10.3779 130 130 Chapter 12: Monsters D&D Basic Rules (Version 1.0). Not for resale. Perm
      132    49        5   0.3551        5  -8.6815 Strength check, freeing itself or another creature within its reach on a success
      337   125        6   0.3454        9  -9.7124 unaligned Armor Class 10 Hit Points 13 (2d10 + 2) Speed 50 ft. STR DEX CON INT W
       11  

In [12]:
# Cell 11.2 — Failure Query 2: Keyword Trap
# "Inspiration" lexically overlaps with Bardic Inspiration, but the question is about the DM Inspiration mechanic
QUERY_2 = "How do you use inspiration?"
br2 = run_baseline(QUERY_2)
rr2 = run_rerank(QUERY_2)
df2 = show_rank_shift(QUERY_2, br2, rr2)
latency_rows.append({
    "query": "Q2: Inspiration mechanic",
    "baseline_retrieval_ms": round(br2["retrieval_ms"]),
    "baseline_llm_ms": round(br2["llm_ms"]),
    "baseline_total_ms": round(br2["retrieval_ms"] + br2["llm_ms"]),
    "rerank_retrieval_ms": round(rr2["retrieval_ms"] + rr2["rerank_ms"]),
    "rerank_llm_ms": round(rr2["llm_ms"]),
    "rerank_total_ms": round(rr2["retrieval_ms"] + rr2["rerank_ms"] + rr2["llm_ms"]),
})


QUERY: How do you use inspiration?
 chunk_id  page  bi_rank bi_score  ce_rank ce_score                                                                     text_preview
      100    38        1   0.3827        3  -1.7666 you can reward another player for good roleplaying, clever thinking, or simply d
       97    37        2   0.3725        2  -1.7607 it when you make an attack roll, saving throw, or ability check. Spending your i
      205    75        3   0.2825        1   0.2420 on the na- ture of your search, the DM might have you make a Wisdom (Perception)
      115    43        4   0.2764       10  -9.7834 of information. 2 Most people scream and run when they see a demon. I stop and t
       85    32        5   0.2665        6  -7.6879 through dedicated study and memorization. You use your Intelligence whenever a s
       80    31        6   0.2605        8  -8.7583 31 31 Chapter 3: Classes D&D Basic Rules (Version 1.0). Not for resale. Permissi
        6     4        7   0.2587

In [13]:
# Cell 11.3 — Failure Query 3: Exception / Corner Case
# General "extra attack" chunks score high, but the specific Rogue sneak attack "once per turn" restriction is buried
QUERY_3 = "Can a Rogue sneak attack twice in one turn?"
br3 = run_baseline(QUERY_3)
rr3 = run_rerank(QUERY_3)
df3 = show_rank_shift(QUERY_3, br3, rr3)
latency_rows.append({
    "query": "Q3: Rogue sneak attack twice",
    "baseline_retrieval_ms": round(br3["retrieval_ms"]),
    "baseline_llm_ms": round(br3["llm_ms"]),
    "baseline_total_ms": round(br3["retrieval_ms"] + br3["llm_ms"]),
    "rerank_retrieval_ms": round(rr3["retrieval_ms"] + rr3["rerank_ms"]),
    "rerank_llm_ms": round(rr3["llm_ms"]),
    "rerank_total_ms": round(rr3["retrieval_ms"] + rr3["rerank_ms"] + rr3["llm_ms"]),
})


QUERY: Can a Rogue sneak attack twice in one turn?
 chunk_id  page  bi_rank bi_score  ce_rank ce_score                                                                     text_preview
       74    29        1   0.3449        1   2.3167 29 29 Chapter 3: Classes D&D Basic Rules (Version 1.0). Not for resale. Permissi
       72    28        2   0.2954        5  -2.0235 28 28 Chapter 3: Classes D&D Basic Rules (Version 1.0). Not for resale. Permissi
      198    73        3   0.2933       10  -5.7477 and swimming. These different modes of movement can be combined with walking, or
       78    30        4   0.2713        2  -0.7162 you have learned enough about the work- ings of magic that you can improvise the
      160    60        5   0.2608        8  -3.4102 also have this bonus, which is incorporated in their stat blocks. The bonus is u
      204    75        6   0.2531        9  -3.8527 friendly creature in attack- ing a creature within 5 feet of you. You feint, dis
      196    72  

In [14]:
# Cell 11.4 — Failure Query 4: Negation / Constraint
# Bi-encoder may confuse "not allowed while incapacitated" with positive action descriptions
QUERY_4 = "What are you not allowed to do while incapacitated?"
br4 = run_baseline(QUERY_4)
rr4 = run_rerank(QUERY_4)
df4 = show_rank_shift(QUERY_4, br4, rr4)
latency_rows.append({
    "query": "Q4: Incapacitated restrictions",
    "baseline_retrieval_ms": round(br4["retrieval_ms"]),
    "baseline_llm_ms": round(br4["llm_ms"]),
    "baseline_total_ms": round(br4["retrieval_ms"] + br4["llm_ms"]),
    "rerank_retrieval_ms": round(rr4["retrieval_ms"] + rr4["rerank_ms"]),
    "rerank_llm_ms": round(rr4["llm_ms"]),
    "rerank_total_ms": round(rr4["retrieval_ms"] + rr4["rerank_ms"] + rr4["llm_ms"]),
})


QUERY: What are you not allowed to do while incapacitated?
 chunk_id  page  bi_rank bi_score  ce_rank ce_score                                                                     text_preview
      432   171        1   0.3816        3  -4.6544 171 171 Appendix A: Conditions D&D Basic Rules (Version 1.0). Not for resale. Pe
      433   171        2   0.3341        1  -1.8104 • A stunned creature is incapacitated (see the condition), can’t move, and can s
      214    79        3   0.3215        4  -5.8738 79 79 Chapter 9: Combat D&D Basic Rules (Version 1.0). Not for resale. Permissio
      183    68        4   0.3170        8  -9.0430 making a Wisdom (Survival) check when the DM calls for it. (The Dungeon Master’s
      218    80        5   0.3004        6  -6.7951 can mount a creature that is within 5 feet of you or dismount. Doing so costs an
      170    64        6   0.2898       10 -10.6688 64 64 Chapter 7: Using Ability Scores D&D Basic Rules (Version 1.0). Not for res
      283

In [15]:
# Cell 11.5 — Failure Query 5: Negation / Prohibition
# Bi-encoder blurs the prohibition with general spellcasting and armor descriptions
QUERY_5 = "Which spells cannot be cast while wearing armor?"
br5 = run_baseline(QUERY_5)
rr5 = run_rerank(QUERY_5)
df5 = show_rank_shift(QUERY_5, br5, rr5)
latency_rows.append({
    "query": "Q5: Spells prohibited in armor",
    "baseline_retrieval_ms": round(br5["retrieval_ms"]),
    "baseline_llm_ms": round(br5["llm_ms"]),
    "baseline_total_ms": round(br5["retrieval_ms"] + br5["llm_ms"]),
    "rerank_retrieval_ms": round(rr5["retrieval_ms"] + rr5["rerank_ms"]),
    "rerank_llm_ms": round(rr5["llm_ms"]),
    "rerank_total_ms": round(rr5["retrieval_ms"] + rr5["rerank_ms"] + rr5["llm_ms"]),
})


QUERY: Which spells cannot be cast while wearing armor?
 chunk_id  page  bi_rank bi_score  ce_rank ce_score                                                                     text_preview
      122    46        1   0.5310        1   2.8061 the worlds of D&D. Armor Proficiency. Anyone can put on a suit of armor or strap
      226    83        2   0.4857        3   1.8137 spell again, you must start over. Range The target of a spell must be within the
      126    47        3   0.4144        9  -5.8617 It does not include leg protection beyond simple greaves that are attached with 
      413   162        4   0.3992        5  -3.2282 has following cleric spells prepared: Cantrips (at will): light, sacred flame, t
      233    86        5   0.3922        7  -5.1397 86 86 Chapter 11: Spells D&D Basic Rules (Version 1.0). Not for resale. Permissi
      395   154        6   0.3837        6  -4.1792 154 154 Chapter 12: Monsters D&D Basic Rules (Version 1.0). Not for resale. Perm
      271   

In [16]:
# Cell 11.6 — Failure Query 6: Buried Chunk (document-specific)
# Concentration rules spread across multiple sections; key "Constitution saving throw" chunk often buried at rank 5-8
QUERY_6 = "What happens to a concentration spell if you take damage?"
br6 = run_baseline(QUERY_6)
rr6 = run_rerank(QUERY_6)
df6 = show_rank_shift(QUERY_6, br6, rr6)
latency_rows.append({
    "query": "Q6: Concentration + damage",
    "baseline_retrieval_ms": round(br6["retrieval_ms"]),
    "baseline_llm_ms": round(br6["llm_ms"]),
    "baseline_total_ms": round(br6["retrieval_ms"] + br6["llm_ms"]),
    "rerank_retrieval_ms": round(rr6["retrieval_ms"] + rr6["rerank_ms"]),
    "rerank_llm_ms": round(rr6["llm_ms"]),
    "rerank_total_ms": round(rr6["retrieval_ms"] + rr6["rerank_ms"] + rr6["llm_ms"]),
})


QUERY: What happens to a concentration spell if you take damage?
 chunk_id  page  bi_rank bi_score  ce_rank ce_score                                                                     text_preview
      225    83        1   0.4541        2   0.9281 creature or an object in a way that can’t be dispelled, because its magic exists
      252    93        2   0.4006        4   0.0604 93 93 Chapter 11: Spells D&D Basic Rules (Version 1.0). Not for resale. Permissi
      262    96        3   0.3635        1   1.7548 damage, or half as much damage on a successful save. The damage can’t reduce the
      281   102        4   0.3373        6  -1.5320 puff of noxious gas from your palm. The creature must succeed on a Constitution 
      216    79        5   0.3331        8  -4.1407 you instantly. When dam- age reduces you to 0 hit points and there is damage rem
      279   102        6   0.3294       10  -8.7156 102 102 Chapter 11: Spells D&D Basic Rules (Version 1.0). Not for resale. Permis
   

In [17]:
# Cell 12: Latency summary table
latency_df = pd.DataFrame(latency_rows)
latency_df.columns = [
    "Query",
    "Baseline Retrieval (ms)", "Baseline LLM (ms)", "Baseline Total (ms)",
    "Rerank Retrieval+Score (ms)", "Rerank LLM (ms)", "Rerank Total (ms)"
]

# Add mean row
mean_row = latency_df.iloc[:, 1:].mean().round().astype(int)
mean_row["Query"] = "MEAN"
latency_df = pd.concat([latency_df, pd.DataFrame([mean_row])], ignore_index=True)

print("Latency Profile (all times in ms):")
print(latency_df.to_string(index=False))

Latency Profile (all times in ms):
                         Query  Baseline Retrieval (ms)  Baseline LLM (ms)  Baseline Total (ms)  Rerank Retrieval+Score (ms)  Rerank LLM (ms)  Rerank Total (ms)
        Q1: Roll a 1 on attack                       11               3541                 3552                          573             4776               5349
      Q2: Inspiration mechanic                       11               2237                 2248                          399             2194               2593
  Q3: Rogue sneak attack twice                       20               1668                 1688                          328             1450               1777
Q4: Incapacitated restrictions                       12               1226                 1238                          371             1222               1593
Q5: Spells prohibited in armor                       45               2242                 2287                          388             2570               2958

## Report Section 1: Architecture Comparison

**Bi-Encoder (Baseline Retrieval)**

A bi-encoder encodes each document chunk and the query independently into fixed-size dense vectors using the same model. Similarity is computed as cosine similarity (or inner product for L2-normalized vectors) between these independent embeddings. This is fast at query time because chunk embeddings are pre-computed and stored in the index. However, the model never sees the query and chunk together, so it cannot capture fine-grained semantic relationships, negations, or conditional qualifiers.

**Cross-Encoder (Re-Ranking)**

A cross-encoder concatenates the query and candidate chunk together and passes them jointly through a transformer. Full self-attention across all tokens in both the query and the chunk lets the model capture nuanced interactions — including negations, exceptions, and semantic dependency. This produces a much more accurate relevance score but requires a separate forward pass for every (query, chunk) pair, so it cannot be precomputed. It is only practical on a small candidate set (e.g., Top 10) after an initial bi-encoder pre-filter.

## Report Section 2: Qualitative & Rank Shift Analysis

*Fill in the observed rank changes and answer quality differences after running all cells (11.1–11.6).*

**Q1 — Keyword Trap: "What happens when you roll a 1 on an attack roll?"**
- Key chunk moved: Rank #__ → Rank #__
- Answer 1 (Baseline) observation: [fill in — did it hallucinate a critical fumble rule?]
- Answer 2 (Re-Ranked) observation: [fill in — did it correctly note 5e has no critical fail mechanic for attacks?]

**Q2 — Keyword Trap: "How do you use inspiration?"**
- Key chunk moved: Rank #__ → Rank #__
- Answer 1 observation: [fill in — did it surface Bardic Inspiration instead of the DM Inspiration mechanic?]
- Answer 2 observation: [fill in]

**Q3 — Exception / Corner Case: "Can a Rogue sneak attack twice in one turn?"**
- Key chunk moved: Rank #__ → Rank #__
- Answer 1 observation: [fill in — did it miss the "once per turn" restriction?]
- Answer 2 observation: [fill in]

**Q4 — Negation / Constraint: "What are you not allowed to do while incapacitated?"**
- Key chunk moved: Rank #__ → Rank #__
- Answer 1 observation: [fill in — did bi-encoder confuse negative framing with positive action text?]
- Answer 2 observation: [fill in]

**Q5 — Negation / Prohibition: "Which spells cannot be cast while wearing armor?"**
- Key chunk moved: Rank #__ → Rank #__
- Answer 1 observation: [fill in]
- Answer 2 observation: [fill in]

**Q6 — Buried Chunk: "What happens to a concentration spell if you take damage?"**
- Key chunk moved: Rank #__ → Rank #__
- Answer 1 observation: [fill in — did the Constitution saving throw rule appear in the top 3?]
- Answer 2 observation: [fill in]

## Report Section 3: Latency Profiling

*Fill in after running Cell 12 and observing the latency table.*

From the latency table (Cell 12):

- **Baseline average total latency:** __ ms (__ ms retrieval + __ ms LLM)
- **Re-ranking average total latency:** __ ms (__ ms retrieval+score + __ ms LLM)
- **Cross-encoder overhead:** approximately __ ms per query (the rerank stage)

Observations:
- The LLM call dominates / does not dominate total latency because [fill in].
- The cross-encoder adds roughly __× the baseline retrieval time.
- [Any outlier queries and why they might be slower — e.g., longer chunks, more complex prompt.]

## Report Section 4: Why Not Run the Cross-Encoder Over the Entire Corpus?

A cross-encoder requires a full transformer forward pass for every (query, document) pair — there is no way to precompute document representations independently. For a corpus of N chunks, this means N separate inference calls per query, each involving self-attention over the concatenated query and chunk tokens.

For a document with ~700 chunks (like the D&D 5e Basic Rules), this is already measurable overhead (~hundreds of milliseconds on CPU). For a real-world corpus of 100,000 documents, running a cross-encoder over every document would take tens of minutes per query — completely impractical for an interactive system.

The two-stage retrieval approach solves this: the bi-encoder pre-filters the corpus down to Top-K candidates (e.g., 10) in milliseconds by comparing precomputed vectors, and the cross-encoder then re-scores only those K pairs precisely. This captures most of the accuracy benefit of the cross-encoder at a fraction of the cost — O(K) cross-encoder calls instead of O(N).